## 0. Налаштування

In [ ]:
!pip install -q \
    "langchain==1.3.11" \
    "langchain-openai==1.3.3" \
    "langchain-community==0.4.2" \
    "langgraph==1.2.7" \
    "langchainhub==0.1.21" \
    "python-dotenv==1.2.2" \
    "wikipedia==1.4.0" \
    "numexpr==2.14.1" \
    "numpy<3" 2>&1 | grep -v "dependency conflicts"

In [ ]:
# import warnings

# warnings.filterwarnings(
#     "ignore",
#     message=".*langchain-community.*",
#     category=DeprecationWarning,
# )

import os
from datetime import datetime
from typing import List, Dict, Any
import numexpr as ne
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.tools import tool
# from langchain_community.utilities import WikipediaAPIWrapper
import re

print("✅ Імпорти виконано.")

✅ Імпорти виконано.


In [ ]:
import os
from google.colab import userdata

api_key = userdata.get("OPENAI_API_KEY")

if not api_key:
    raise ValueError("Секрет OPENAI не знайдено.")

os.environ["OPENAI_API_KEY"] = api_key

print("✅ Ключ успішно завантажено")
print("Ключ встановлено:", bool(os.environ.get("OPENAI_API_KEY")))

✅ Ключ успішно завантажено
Ключ встановлено: True


## 1. Бази даних і структури

In [ ]:
RECIPES_DB = {
    "борщ": {
        "ingredients": [
            "буряк — 300 г",
            "капуста — 300 г",
            "картопля — 400 г",
            "морква — 150 г",
            "цибуля — 100 г",
            "томатна паста — 2 ст.л.",
            "вода — 2 л",
            "олія — 1 ст.л.",
            "сіль — за смаком",
        ],
        "time_minutes": 90,
        "servings": 6,
        "instructions": (
            "1. Підготуйте та наріжте овочі. "
            "2. Відваріть картоплю й капусту. "
            "3. Додайте тушковані буряк, моркву, цибулю та томатну пасту. "
            "4. Варіть до готовності."
        ),
        "tags": [
            "перша страва",
            "суп",
            "обід",
            "українська кухня",
            "вегетаріанське",
            "овочеве",
        ],
        "gluten_free": True,
    },

    "омлет": {
        "ingredients": [
            "яйця — 4 шт.",
            "молоко — 100 мл",
            "сіль — за смаком",
            "олія — 1 ч.л.",
        ],
        "time_minutes": 10,
        "servings": 2,
        "instructions": (
            "1. Збийте яйця з молоком і сіллю. "
            "2. Розігрійте пательню з олією. "
            "3. Вилийте суміш і готуйте під кришкою."
        ),
        "tags": [
            "сніданок",
            "швидко",
            "білкове",
            "вегетаріанське",
        ],
        "gluten_free": True,
    },

    "курка з картоплею": {
        "ingredients": [
            "куряче філе — 500 г",
            "картопля — 700 г",
            "морква — 150 г",
            "олія — 2 ст.л.",
            "сіль — за смаком",
            "перець — за смаком",
        ],
        "time_minutes": 55,
        "servings": 4,
        "instructions": (
            "1. Наріжте курку, картоплю та моркву. "
            "2. Додайте олію, сіль і перець. "
            "3. Запікайте при 190°C приблизно 40–45 хвилин."
        ),
        "tags": [
            "основна страва",
            "обід",
            "вечеря",
            "білкове",
            "духовка",
            "курка",
        ],
        "gluten_free": True,
    },

    "вареники з картоплею": {
        "ingredients": [
            "борошно — 500 г",
            "вода — 250 мл",
            "яйця — 1 шт.",
            "картопля — 700 г",
            "цибуля — 150 г",
            "олія — 1 ст.л.",
            "сіль — за смаком",
        ],
        "time_minutes": 80,
        "servings": 5,
        "instructions": (
            "1. Замісіть тісто з борошна, води, яйця та солі. "
            "2. Приготуйте начинку з картоплі й цибулі. "
            "3. Сформуйте вареники. "
            "4. Варіть у підсоленій воді 4–5 хвилин після спливання."
        ),
        "tags": [
            "основна страва",
            "обід",
            "українська кухня",
            "вегетаріанське",
            "тісто",
        ],
        "gluten_free": False,
    },

    "млинці": {
        "ingredients": [
            "борошно — 250 г",
            "молоко — 500 мл",
            "яйця — 2 шт.",
            "цукор — 30 г",
            "сіль — 1 щіпка",
            "олія — 2 ст.л.",
        ],
        "time_minutes": 30,
        "servings": 4,
        "instructions": (
            "1. Змішайте яйця, молоко, цукор і сіль. "
            "2. Поступово додайте борошно. "
            "3. Додайте трохи олії. "
            "4. Смажте тонкі млинці з обох боків."
        ),
        "tags": [
            "сніданок",
            "десерт",
            "солодке",
            "вегетаріанське",
            "швидко",
        ],
        "gluten_free": False,
    },

    "овочевий салат": {
        "ingredients": [
            "помідори — 250 г",
            "огірки — 200 г",
            "болгарський перець — 150 г",
            "листя салату — 80 г",
            "олія — 1 ст.л.",
            "сіль — за смаком",
        ],
        "time_minutes": 10,
        "servings": 2,
        "instructions": (
            "1. Помийте та наріжте овочі. "
            "2. Додайте листя салату. "
            "3. Заправте олією та посоліть."
        ),
        "tags": [
            "салат",
            "швидко",
            "вегетаріанське",
            "легка страва",
            "овочеве",
            "обід",
            "вечеря",
        ],
        "gluten_free": True,
    },

    "рибний суп": {
        "ingredients": [
            "риба — 400 г",
            "картопля — 300 г",
            "морква — 100 г",
            "цибуля — 100 г",
            "вода — 1.5 л",
            "сіль — за смаком",
            "зелень — 20 г",
        ],
        "time_minutes": 45,
        "servings": 4,
        "instructions": (
            "1. Наріжте овочі та відваріть їх до напівготовності. "
            "2. Додайте шматочки риби. "
            "3. Варіть ще 10–15 хвилин. "
            "4. Додайте сіль і зелень."
        ),
        "tags": [
            "перша страва",
            "суп",
            "рибне",
            "обід",
            "білкове",
        ],
        "gluten_free": True,
    },

    "овочеве рагу": {
        "ingredients": [
            "кабачок — 300 г",
            "картопля — 400 г",
            "морква — 150 г",
            "болгарський перець — 150 г",
            "помідори — 250 г",
            "олія — 2 ст.л.",
            "сіль — за смаком",
        ],
        "time_minutes": 40,
        "servings": 4,
        "instructions": (
            "1. Наріжте всі овочі. "
            "2. Злегка протушкуйте моркву та картоплю. "
            "3. Додайте інші овочі. "
            "4. Тушкуйте під кришкою до готовності."
        ),
        "tags": [
            "основна страва",
            "обід",
            "вечеря",
            "вегетаріанське",
            "овочеве",
            "тушковане",
        ],
        "gluten_free": True,
    },

    "яблучний пиріг": {
        "ingredients": [
            "борошно — 250 г",
            "яблука — 500 г",
            "яйця — 3 шт.",
            "цукор — 150 г",
            "масло — 100 г",
            "розпушувач — 10 г",
        ],
        "time_minutes": 60,
        "servings": 8,
        "instructions": (
            "1. Збийте яйця з цукром. "
            "2. Додайте розтоплене масло, борошно та розпушувач. "
            "3. Додайте нарізані яблука. "
            "4. Випікайте при 180°C приблизно 40 хвилин."
        ),
        "tags": [
            "десерт",
            "випічка",
            "солодке",
            "духовка",
            "вегетаріанське",
        ],
        "gluten_free": False,
    },

    "рисова каша": {
        "ingredients": [
            "рис — 180 г",
            "молоко — 500 мл",
            "вода — 250 мл",
            "цукор — 30 г",
            "сіль — 1 щіпка",
            "масло — 20 г",
        ],
        "time_minutes": 35,
        "servings": 3,
        "instructions": (
            "1. Промийте рис. "
            "2. Варіть його у воді 10 хвилин. "
            "3. Додайте молоко, цукор і сіль. "
            "4. Варіть до м'якості та додайте масло."
        ),
        "tags": [
            "сніданок",
            "каша",
            "солодке",
            "вегетаріанське",
        ],
        "gluten_free": True,
    },

    "гречка з куркою": {
        "ingredients": [
            "гречка — 200 г",
            "куряче філе — 400 г",
            "морква — 100 г",
            "цибуля — 100 г",
            "олія — 1 ст.л.",
            "сіль — за смаком",
        ],
        "time_minutes": 40,
        "servings": 4,
        "instructions": (
            "1. Наріжте та приготуйте куряче філе. "
            "2. Додайте моркву й цибулю. "
            "3. Окремо відваріть гречку. "
            "4. Змішайте всі інгредієнти."
        ),
        "tags": [
            "основна страва",
            "обід",
            "вечеря",
            "білкове",
            "швидко",
            "курка",
        ],
        "gluten_free": True,
    },

    "салат з тунцем": {
        "ingredients": [
            "тунець — 160 г",
            "яйця — 2 шт.",
            "огірки — 150 г",
            "помідори — 200 г",
            "листя салату — 80 г",
            "олія — 1 ст.л.",
        ],
        "time_minutes": 15,
        "servings": 2,
        "instructions": (
            "1. Відваріть і наріжте яйця. "
            "2. Наріжте овочі. "
            "3. Додайте тунець і листя салату. "
            "4. Заправте олією."
        ),
        "tags": [
            "салат",
            "обід",
            "вечеря",
            "білкове",
            "швидко",
            "рибне",
            "легка страва",
        ],
        "gluten_free": True,
    },

    "паста з куркою у вершковому соусі": {
        "ingredients": [
            "паста — 250 г",
            "куряче філе — 350 г",
            "вершки — 200 мл",
            "часник — 2 зубчики",
            "олія — 1 ст.л.",
            "сіль — за смаком",
            "перець — за смаком",
        ],
        "time_minutes": 30,
        "servings": 3,
        "instructions": (
            "1. Відваріть пасту відповідно до інструкції на упаковці. "
            "2. Наріжте куряче філе та обсмажте на олії. "
            "3. Додайте подрібнений часник і вершки. "
            "4. Тушкуйте соус кілька хвилин. "
            "5. Додайте пасту, перемішайте та прогрійте."
        ),
        "tags": [
            "основна страва",
            "обід",
            "вечеря",
            "паста",
            "курка",
            "білкове",
            "швидко",
        ],
        "gluten_free": False,
    },

    "запечений лосось з овочами": {
        "ingredients": [
            "лосось — 350 г",
            "кабачок — 200 г",
            "болгарський перець — 150 г",
            "помідори — 200 г",
            "олія — 1 ст.л.",
            "лимон — 0.5 шт.",
            "сіль — за смаком",
            "перець — за смаком",
        ],
        "time_minutes": 35,
        "servings": 2,
        "instructions": (
            "1. Наріжте овочі та викладіть у форму для запікання. "
            "2. Додайте олію, сіль і перець. "
            "3. Покладіть зверху лосось і скропіть лимонним соком. "
            "4. Запікайте при 190°C приблизно 20–25 хвилин."
        ),
        "tags": [
            "основна страва",
            "обід",
            "вечеря",
            "рибне",
            "білкове",
            "духовка",
            "легка страва",
        ],
        "gluten_free": True,
    },

    "рис з куркою та овочами": {
        "ingredients": [
            "рис — 200 г",
            "куряче філе — 400 г",
            "морква — 100 г",
            "болгарський перець — 150 г",
            "кабачок — 200 г",
            "олія — 1 ст.л.",
            "сіль — за смаком",
            "перець — за смаком",
        ],
        "time_minutes": 35,
        "servings": 4,
        "instructions": (
            "1. Відваріть рис до готовності. "
            "2. Наріжте куряче філе та обсмажте на олії. "
            "3. Додайте нарізані овочі та готуйте до м'якості. "
            "4. Додайте рис, сіль і перець та перемішайте."
        ),
        "tags": [
            "основна страва",
            "обід",
            "вечеря",
            "курка",
            "білкове",
            "швидко",
        ],
        "gluten_free": True,
    },

    "шакшука": {
        "ingredients": [
            "яйця — 4 шт.",
            "помідори — 400 г",
            "болгарський перець — 150 г",
            "цибуля — 100 г",
            "часник — 2 зубчики",
            "олія — 1 ст.л.",
            "сіль — за смаком",
        ],
        "time_minutes": 25,
        "servings": 2,
        "instructions": (
            "1. Наріжте цибулю та перець і обсмажте на олії. "
            "2. Додайте помідори й часник та тушкуйте кілька хвилин. "
            "3. Зробіть заглиблення у соусі та розбийте в них яйця. "
            "4. Готуйте під кришкою до бажаної готовності яєць."
        ),
        "tags": [
            "сніданок",
            "швидко",
            "вегетаріанське",
            "білкове",
            "яйця",
        ],
        "gluten_free": True,
    },

    "крем-суп з кабачка": {
        "ingredients": [
            "кабачок — 500 г",
            "картопля — 250 г",
            "морква — 100 г",
            "цибуля — 100 г",
            "вершки — 150 мл",
            "вода — 800 мл",
            "сіль — за смаком",
            "олія — 1 ст.л.",
        ],
        "time_minutes": 35,
        "servings": 4,
        "instructions": (
            "1. Наріжте овочі. "
            "2. Відваріть їх у воді до м'якості. "
            "3. Подрібніть овочі блендером до однорідності. "
            "4. Додайте вершки, сіль і прогрійте суп ще кілька хвилин."
        ),
        "tags": [
            "перша страва",
            "суп",
            "обід",
            "вегетаріанське",
            "овочеве",
            "легка страва",
        ],
        "gluten_free": True,
    },

    "спагеті з томатним соусом": {
        "ingredients": [
            "спагеті — 250 г",
            "помідори — 400 г",
            "томатна паста — 1 ст.л.",
            "часник — 2 зубчики",
            "олія — 1 ст.л.",
            "сіль — за смаком",
            "зелень — 20 г",
        ],
        "time_minutes": 25,
        "servings": 3,
        "instructions": (
            "1. Відваріть спагеті до готовності. "
            "2. На олії коротко обсмажте часник. "
            "3. Додайте помідори та томатну пасту і тушкуйте 10 хвилин. "
            "4. Змішайте соус зі спагеті та додайте зелень."
        ),
        "tags": [
            "основна страва",
            "обід",
            "вечеря",
            "паста",
            "вегетаріанське",
            "швидко",
        ],
        "gluten_free": False,
    },

    "запечені овочі з фетою": {
        "ingredients": [
            "кабачок — 300 г",
            "болгарський перець — 200 г",
            "помідори — 300 г",
            "фета — 150 г",
            "олія — 1.5 ст.л.",
            "сіль — за смаком",
            "зелень — 20 г",
        ],
        "time_minutes": 35,
        "servings": 3,
        "instructions": (
            "1. Наріжте овочі та викладіть у форму. "
            "2. Додайте олію та сіль. "
            "3. Запікайте при 190°C приблизно 20 хвилин. "
            "4. Додайте фету і запікайте ще 5–10 хвилин. "
            "5. Перед подачею додайте зелень."
        ),
        "tags": [
            "основна страва",
            "обід",
            "вечеря",
            "вегетаріанське",
            "овочеве",
            "духовка",
            "легка страва",
        ],
        "gluten_free": True,
    },

    "бананові панкейки": {
        "ingredients": [
            "банан — 1 шт. (приблизно 120 г)",
            "яйця — 2 шт.",
            "борошно — 100 г",
            "молоко — 100 мл",
            "олія — 1 ч.л.",
        ],
        "time_minutes": 20,
        "servings": 2,
        "instructions": (
            "1. Розімніть банан виделкою. "
            "2. Додайте яйця та молоко і перемішайте. "
            "3. Додайте борошно та перемішайте до однорідності. "
            "4. Смажте невеликі панкейки на пательні з обох боків."
        ),
        "tags": [
            "сніданок",
            "десерт",
            "солодке",
            "швидко",
            "вегетаріанське",
        ],
        "gluten_free": False,
    },
}

UNIT_CONVERSIONS = {
    # Склянки → грами для конкретних продуктів
    ("склянка", "г", "борошно"): 150,
    ("склянка", "г", "цукор"): 200,
    ("склянка", "г", "рис"): 180,
    ("склянка", "г", "гречка"): 170,
    ("склянка", "г", "вівсянка"): 90,

    # Склянки → мілілітри
    ("склянка", "мл", ""): 240,

    # Столові ложки → мілілітри
    ("ст.л.", "мл", ""): 15,

    # Чайні ложки → мілілітри
    ("ч.л.", "мл", ""): 5,

    # Столові ложки → грами
    ("ст.л.", "г", "борошно"): 10,
    ("ст.л.", "г", "цукор"): 20,
    ("ст.л.", "г", "масло"): 15,
    ("ст.л.", "г", "олія"): 14,

    # Чайні ложки → грами
    ("ч.л.", "г", "сіль"): 7,
    ("ч.л.", "г", "цукор"): 5,
    ("ч.л.", "г", "олія"): 5,
}


SUBSTITUTIONS = {
    "яйця": [
        {
            "замінник": "банан — половина стиглого банана замість 1 яйця",
            "для": "солодка випічка, панкейки",
            "примітка": "додає солодкість, вологість і легкий банановий смак",
        },
        {
            "замінник": "льняне насіння — 1 ст.л. меленого насіння + 3 ст.л. води",
            "для": "випічка, панкейки",
            "примітка": "залиш суміш приблизно на 10 хвилин до загущення",
        },
        {
            "замінник": "аквафаба — 3 ст.л. замість 1 яйця",
            "для": "меренги, муси, легка випічка",
            "примітка": "аквафаба — це рідина з-під консервованого нуту",
        },
    ],

    "молоко": [
        {
            "замінник": "вівсяний напій",
            "для": "каші, млинці, випічка, соуси",
            "примітка": "має нейтральний смак; при безглютеновій дієті перевір маркування",
        },
        {
            "замінник": "соєвий напій",
            "для": "омлети, випічка, соуси, каші",
            "примітка": "за текстурою часто близький до звичайного молока, але містить сою",
        },
        {
            "замінник": "мигдальний напій",
            "для": "каші, десерти, випічка",
            "примітка": "може мати легкий горіховий смак",
        },
        {
            "замінник": "кокосове молоко",
            "для": "каші, десерти, соуси",
            "примітка": "має виражений кокосовий смак і може бути жирнішим",
        },
    ],

    "вершки": [
        {
            "замінник": "рослинні кулінарні вершки",
            "для": "вершкові соуси, паста, крем-супи",
            "примітка": "найближча заміна за текстурою; обирай продукт саме для готування",
        },
        {
            "замінник": "кокосові вершки",
            "для": "соуси, супи, десерти",
            "примітка": "добре загущують, але можуть додати кокосовий смак",
        },
        {
            "замінник": "молоко або рослинний напій + трохи крохмалю",
            "для": "соуси та крем-супи",
            "примітка": "сама рідина значно рідша за вершки, тому для густоти потрібен крохмаль",
        },
    ],

    "борошно": [
        {
            "замінник": "рисове борошно",
            "для": "млинці, печиво, безглютенова випічка",
            "примітка": "може зробити готовий виріб більш крихким",
        },
        {
            "замінник": "вівсяне борошно",
            "для": "млинці, панкейки, випічка",
            "примітка": "для безглютенового рецепта потрібно перевірити відповідне маркування",
        },
        {
            "замінник": "кукурудзяне борошно",
            "для": "коржі, панкейки, несолодка випічка",
            "примітка": "змінює структуру, колір і смак тіста",
        },
    ],

    "масло": [
        {
            "замінник": "рослинна олія",
            "для": "випічка, смаження",
            "примітка": "100 г масла можна орієнтовно замінити приблизно 80 мл олії",
        },
        {
            "замінник": "кокосова олія",
            "для": "десерти, випічка",
            "примітка": "може додати кокосовий аромат",
        },
        {
            "замінник": "яблучне пюре",
            "для": "солодка випічка",
            "примітка": "зменшує жирність і робить випічку більш вологою",
        },
    ],

    "цукор": [
        {
            "замінник": "мед",
            "для": "десерти, напої, випічка",
            "примітка": "мед додає рідину, тому в деяких рецептах потрібно зменшити інші рідини",
        },
        {
            "замінник": "стиглий банан",
            "для": "каші, панкейки, солодка випічка",
            "примітка": "додає власний смак і змінює текстуру",
        },
        {
            "замінник": "еритритол",
            "для": "напої, десерти, деякі види випічки",
            "примітка": "солодкість відрізняється від цукру, тому пропорцію потрібно коригувати",
        },
    ],

    "сметана": [
        {
            "замінник": "натуральний йогурт",
            "для": "соуси, заправки, випічка",
            "примітка": "може бути менш жирним і трохи кислішим",
        },
        {
            "замінник": "грецький йогурт",
            "для": "соуси, заправки, подача до страв",
            "примітка": "має густу текстуру, близьку до сметани",
        },
        {
            "замінник": "рослинний йогурт без цукру",
            "для": "соуси та заправки",
            "примітка": "підійде для безлактозного варіанта, якщо смак нейтральний",
        },
    ],

    "фета": [
        {
            "замінник": "бринза",
            "для": "салати, запечені овочі",
            "примітка": "може бути солонішою, тому обережно додавай сіль",
        },
        {
            "замінник": "козячий сир",
            "для": "салати, запечені овочі",
            "примітка": "має більш виражений смак",
        },
        {
            "замінник": "рослинний сир типу фета",
            "для": "салати, запечені овочі",
            "примітка": "можна використати для безлактозного або веганського варіанта",
        },
    ],

    "рис": [
        {
            "замінник": "гречка",
            "для": "гарнір, страви з куркою та овочами",
            "примітка": "має більш виражений смак і іншу текстуру",
        },
        {
            "замінник": "кіноа",
            "для": "гарніри, салати",
            "примітка": "текстура та час приготування відрізняються від рису",
        },
        {
            "замінник": "булгур",
            "для": "гарнір, страви з овочами",
            "примітка": "містить глютен",
        },
    ],

    "куряче філе": [
        {
            "замінник": "філе індички",
            "для": "запікання, смаження, страви з рисом або пастою",
            "примітка": "найближча заміна за способом приготування",
        },
        {
            "замінник": "тофу",
            "для": "страви з овочами, рисом, салати",
            "примітка": "вегетаріанська альтернатива з іншою текстурою та смаком",
        },
    ],

    "лосось": [
        {
            "замінник": "форель",
            "для": "запікання, смаження",
            "примітка": "дуже близька за жирністю та способом приготування",
        },
        {
            "замінник": "дорадо",
            "для": "запікання",
            "примітка": "менш жирна риба, тому результат буде трохи іншим",
        },
        {
            "замінник": "хек",
            "для": "запікання, тушкування",
            "примітка": "значно менш жирний за лосось",
        },
    ],

    "помідори": [
        {
            "замінник": "консервовані томати",
            "для": "соуси, супи, шакшука",
            "примітка": "добре підходять для термічної обробки",
        },
        {
            "замінник": "томатна пасата",
            "для": "соуси, супи",
            "примітка": "має більш однорідну текстуру",
        },
    ],
}



print(f"✅ База: {len(RECIPES_DB)} рецептів, {len(SUBSTITUTIONS)} замінників, {len(SUBSTITUTIONS)}")

✅ База: 20 рецептів, 12 замінників, 12


## 2. Інструменти (@tool)

In [ ]:
from langchain_core.tools import tool
import re


@tool
def recipe_search(query: str) -> str:
    """
    Шукає рецепти за назвою, інгредієнтами, тегами або харчовими обмеженнями.

    Використовуй цей інструмент, коли користувач:
    - просить рецепт конкретної страви;
    - хоче знайти страву з наявних продуктів;
    - шукає страву певної категорії;
    - просить рецепт без глютену.

    Повертає до трьох найбільш релевантних рецептів:
    назву, інгредієнти, час приготування, кількість порцій
    та коротку інструкцію.
    """
    if not isinstance(query, str) or not query.strip():
        return "Запит порожній. Вкажи назву страви, інгредієнти або категорію."

    query_lower = query.lower().strip()
    query_words = [
        word
        for word in re.findall(r"[а-яіїєґa-z0-9'-]+", query_lower)
        if len(word) > 2
    ]

    results = []

    for name, recipe in RECIPES_DB.items():
        score = 0
        match_reasons = []

        # Точний або частковий збіг за назвою
        if query_lower in name.lower() or name.lower() in query_lower:
            score += 5
            match_reasons.append("назва")

        # Пошук за інгредієнтами
        matched_ingredients = []
        for ingredient in recipe["ingredients"]:
            ingredient_lower = ingredient.lower()

            if any(
                word in ingredient_lower or ingredient_lower in word
                for word in query_words
            ):
                matched_ingredients.append(ingredient)

        if matched_ingredients:
            score += len(matched_ingredients) * 2
            match_reasons.append(
                f"інгредієнти: {', '.join(matched_ingredients)}"
            )

        # Пошук за тегами
        matched_tags = []
        for tag in recipe["tags"]:
            tag_lower = tag.lower()

            if any(
                word in tag_lower or tag_lower in word
                for word in query_words
            ):
                matched_tags.append(tag)

        if matched_tags:
            score += len(matched_tags)
            match_reasons.append(f"теги: {', '.join(matched_tags)}")

        # Окрема перевірка запиту без глютену
        gluten_free_request = (
            "без глютену" in query_lower
            or "безглютен" in query_lower
            or "глютен free" in query_lower
        )

        if gluten_free_request:
            if recipe["gluten_free"]:
                score += 4
                match_reasons.append("без глютену")
            else:
                # Рецепти з глютеном не повертаємо
                continue

        if score > 0:
            results.append(
                {
                    "name": name,
                    "recipe": recipe,
                    "score": score,
                    "reason": "; ".join(match_reasons),
                }
            )

    if not results:
        return (
            f"У локальній базі не знайдено рецептів за запитом «{query}». "
            "Спробуй назвати інший продукт, страву або категорію."
        )

    # Найбільш релевантні рецепти показуємо першими
    results.sort(key=lambda item: item["score"], reverse=True)

    output = []

    for item in results[:3]:
        name = item["name"]
        recipe = item["recipe"]

        output.append(f"🍽️ {name.capitalize()}")
        output.append(
            f"Інгредієнти: {', '.join(recipe['ingredients'])}"
        )
        output.append(
            f"Час: {recipe['time_minutes']} хвилин | "
            f"Порцій: {recipe['servings']}"
        )
        output.append(
            f"Без глютену: {'так' if recipe['gluten_free'] else 'ні'}"
        )
        output.append(f"Знайдено за: {item['reason']}")
        output.append(f"Приготування: {recipe['instructions']}")
        output.append("")

    return "\n".join(output).strip()


@tool
def unit_converter(query: str) -> str:
    """
    Конвертує кулінарні одиниці вимірювання на основі локальної таблиці.

    Використовуй цей інструмент, коли користувач просить:
    - перевести склянки в грами або мілілітри;
    - перевести столові чи чайні ложки в грами або мілілітри;
    - визначити вагу певної кількості продукту.

    Приклади:
    - "скільки грамів у склянці борошна";
    - "2 склянки цукру — це скільки грамів";
    - "3 столові ложки — це скільки мілілітрів".

    Повертає результат лише для конвертацій, які є в UNIT_CONVERSIONS.
    """
    if not isinstance(query, str) or not query.strip():
        return "Запит порожній. Вкажи кількість, одиницю та потрібний формат."

    query_lower = query.lower().strip()

    # Визначення кількості
    numbers = re.findall(r"\d+(?:[.,]\d+)?", query_lower)

    try:
        amount = float(numbers[0].replace(",", ".")) if numbers else 1.0
    except (ValueError, IndexError):
        return "Не вдалося визначити кількість. Спробуй вказати число, наприклад 2."

    if amount <= 0:
        return "Кількість повинна бути більшою за нуль."

    # Визначення початкової одиниці
    from_unit = None

    if "склянк" in query_lower:
        from_unit = "склянка"
    elif (
        "ст.л." in query_lower
        or "столов" in query_lower
        or "столова лож" in query_lower
    ):
        from_unit = "ст.л."
    elif (
        "ч.л." in query_lower
        or "чайн" in query_lower
        or "чайна лож" in query_lower
    ):
        from_unit = "ч.л."

    # Визначення цільової одиниці
    to_unit = None

    if "грам" in query_lower or re.search(r"\bг\b", query_lower):
        to_unit = "г"
    elif "мілілітр" in query_lower or re.search(r"\bмл\b", query_lower):
        to_unit = "мл"

    # Визначення продукту
    product = ""

    known_products = [
        "борошно",
        "цукор",
        "рис",
        "гречка",
        "сіль",
        "масло",
    ]

    for known_product in known_products:
        if known_product in query_lower:
            product = known_product
            break

    if not from_unit:
        return (
            "Не вдалося визначити початкову одиницю. "
            "Використай слова «склянка», «столова ложка» або «чайна ложка»."
        )

    if not to_unit:
        return (
            "Не вдалося визначити цільову одиницю. "
            "Вкажи, чи потрібні грами або мілілітри."
        )

    specific_key = (from_unit, to_unit, product)
    universal_key = (from_unit, to_unit, "")

    if product and specific_key in UNIT_CONVERSIONS:
        conversion_value = UNIT_CONVERSIONS[specific_key]
        result = amount * conversion_value

        return (
            f"{amount:g} {from_unit} продукту «{product}» "
            f"≈ {result:.1f} {to_unit}."
        )

    if universal_key in UNIT_CONVERSIONS:
        conversion_value = UNIT_CONVERSIONS[universal_key]
        result = amount * conversion_value

        return f"{amount:g} {from_unit} ≈ {result:.1f} {to_unit}."

    if not product and to_unit == "г":
        return (
            "Для переведення в грами потрібно вказати продукт, "
            "оскільки різні продукти мають різну щільність."
        )

    return (
        f"У локальній таблиці немає конвертації з «{from_unit}» "
        f"у «{to_unit}» для продукту «{product or 'не вказано'}»."
    )


@tool
def substitution_finder(ingredient: str) -> str:
    """
    Шукає кулінарні замінники для конкретного інгредієнта.

    Використовуй цей інструмент, коли користувач:
    - питає, чим замінити інгредієнт;
    - не має певного продукту;
    - хоче адаптувати рецепт;
    - має харчове обмеження.

    Приклади:
    - "чим замінити яйця";
    - "що використати замість молока";
    - "у мене немає сметани".

    Повертає доступні заміни, сферу використання та важливі примітки.
    """
    if not isinstance(ingredient, str) or not ingredient.strip():
        return "Вкажи назву інгредієнта, який потрібно замінити."

    ingredient_lower = ingredient.lower().strip()

    found_key = None

    for key in SUBSTITUTIONS:
        if key in ingredient_lower or ingredient_lower in key:
            found_key = key
            break

    if not found_key:
        return (
            f"У локальній базі не знайдено замінника для «{ingredient}». "
            "Спробуй вказати коротку назву продукту, наприклад «молоко»."
        )

    result = [f"Замінники для «{found_key}»:"]

    for substitute in SUBSTITUTIONS[found_key]:
        result.append(
            f"• {substitute['замінник']}. "
            f"Підходить для: {substitute['для']}. "
            f"Примітка: {substitute['примітка']}."
        )

    return "\n".join(result)


# Список інструментів, які будуть доступні агенту
tools = [
    recipe_search,
    unit_converter,
    substitution_finder,
]

print(f"✅ Інструменти ChefBot створено: {len(tools)}")
for current_tool in tools:
    print(f"— {current_tool.name}")

✅ Інструменти ChefBot створено: 3
— recipe_search
— unit_converter
— substitution_finder


## 3. Модель

In [ ]:
chat_model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.3,
    max_tokens=512,
    timeout=30,
)

test = chat_model.invoke("Привіт! Одним реченням підтвердь, що ти готовий працювати як ChefBot.")
print("✅ Модель ОК:", test.content)

✅ Модель ОК: Привіт! Я готовий працювати як ChefBot і допомогти тобі з рецептами та кулінарними порадами!


## 4. Агент

In [ ]:
SYSTEM_PROMPT = """
Ти — ChefBot, дружній персональний кулінарний AI-асистент.
Твоя задача — допомагати користувачу:
- знаходити рецепти;
- вирішувати, що приготувати з наявних продуктів;
- знаходити рецепти за категорією або харчовими обмеженнями;
- конвертувати кулінарні одиниці вимірювання;
- підбирати заміни інгредієнтів;
- відповідати на прості запитання, пов'язані з приготуванням їжі.

Відповідай українською мовою, якщо користувач не попросив іншу мову.
У тебе є спеціальні інструменти:
1. recipe_search — пошук рецептів.
2. unit_converter — конвертація кулінарних одиниць вимірювання.
3. substitution_finder — пошук замін інгредієнтів.

=== ГОЛОВНЕ ПРАВИЛО ===
Якщо для запиту існує відповідний інструмент, ОБОВ'ЯЗКОВО використовуй його.
Не замінюй роботу інструмента власними знаннями, припущеннями або приблизними оцінками.
Спочатку отримай результат від відповідного інструмента, а потім сформуй зрозумілу відповідь користувачу.

=== 1. ПОШУК РЕЦЕПТІВ ===

ОБОВ'ЯЗКОВО використовуй recipe_search, якщо користувач:
- просить рецепт конкретної страви;
- питає, що можна приготувати;
- називає продукти й хоче отримати ідею страви;
- шукає сніданок, обід, вечерю або перекус;
- просить швидку, просту або іншу категорію страв;
- просить рецепт із певним харчовим обмеженням, наприклад без глютену.
Приклади запитів, для яких потрібно використовувати recipe_search:
"Що приготувати з курки та картоплі?"
"Дай рецепт омлету."
"Хочу швидкий сніданок."
"Що можна зробити з тунцем?"
"Дай рецепт без глютену."
"Порадь просту вечерю."
Не вигадуй конкретний рецепт, якщо його немає в результатах recipe_search.
Якщо recipe_search не знайшов відповідного рецепта:
- чесно скажи, що в поточній базі відповідного рецепта немає;
- не вигадуй рецепт від себе;
- за потреби запропонуй користувачу змінити умови пошуку.

=== 2. КОНВЕРТАЦІЯ ОДИНИЦЬ ===

ОБОВ'ЯЗКОВО використовуй unit_converter,
якщо користувач просить перевести кулінарні одиниці вимірювання.
Наприклад:
- склянки у грами;
- ложки у грами;
- ложки у мілілітри;
- грами у склянки;
- мілілітри у ложки;
- або іншу конвертацію, яку підтримує інструмент.

Особливо звертай увагу на слова: "склянка", "склянки", "ложка", "ложки",
"г", "грам", "грами", "мл", "мілілітри".
Не виконуй такі розрахунки самостійно.
Не відповідай приблизними значеннями зі своєї пам'яті.
Не використовуй формулювання: "зазвичай..." "приблизно..." "орієнтовно..."
як заміну виклику unit_converter.

Приклад: Запит: "Скільки грамів борошна у 2 склянках?"
Правильна поведінка:
1. Викликати unit_converter.
2. Отримати результат.
3. Передати користувачу значення, яке повернув інструмент.

Неправильна поведінка: відповісти зі своїх знань без виклику unit_converter.
Не використовуй власні знання про щільність продукту для отримання приблизної відповіді.

Якщо unit_converter повідомляє, що потрібної конвертації або продукту немає в базі:
- так і скажи користувачу;
- не вигадуй власне значення.

=== 3. ЗАМІНА ІНГРЕДІЄНТІВ ===
ОБОВ'ЯЗКОВО використовуй substitution_finder, якщо користувач:
- питає, чим замінити певний продукт;
- не має потрібного інгредієнта;
- не вживає певний продукт і хоче альтернативу.
Приклади: "Чим замінити вершки?" "У мене немає яєць. Чим їх замінити?"
"Я не вживаю молоко. Що можна використати замість нього?"
Не вигадуй конкретні заміни, якщо substitution_finder їх не повернув.
Якщо інструмент не знайшов заміну:
- чесно повідом про це;
- не підставляй альтернативу зі своєї пам'яті.

=== 4. КОМБІНОВАНІ ЗАПИТИ ===
Один запит користувача може потребувати кількох інструментів.
Наприклад: "Що приготувати з курки та картоплі і чим можна замінити вершки?"
У такому випадку можна використати:
1. recipe_search;
2. substitution_finder.
Не обмежуйся одним інструментом, якщо для виконання всіх умов запиту потрібні кілька.

=== 5. КОНТЕКСТ ДІАЛОГУ ===

Враховуй попередні повідомлення поточної розмови.
Якщо користувач після отриманого рецепта пише: "А чим це замінити?"
"Перерахуй." "Що далі?" "А без цього можна?"
використовуй контекст попередніх повідомлень, щоб зрозуміти, про який рецепт або
інгредієнт ідеться.
Не змушуй користувача повторювати інформацію, якщо вона вже є в поточній історії діалогу.

=== 6. ЯКЩО ДАНИХ НЕДОСТАТНЬО ===

Якщо без додаткової інформації неможливо правильно виконати запит,
постав одне коротке уточнювальне запитання.
Наприклад: "Який саме продукт потрібно перевести у грами?"
 "Який інгредієнт ви хочете замінити?" Не вигадуй відсутні дані.

 === 7. ТОЧНІСТЬ ===

 Розрізняй інформацію, отриману від інструментів, і загальні кулінарні пояснення.
 Точні значення, які повернув інструмент, не змінюй на власний розсуд.
 Якщо інструмент не має потрібних даних:
 - скажи про це;
 - не створюй точне значення самостійно.

 Якщо користувач питає, скільки грамів, мілілітрів, штук або іншої кількості
 інгредієнта потрібно для конкретного рецепта, використовуй тільки дані,
 які є у знайденому рецепті.

Не вигадуй кількість інгредієнтів зі своїх знань.

Не використовуй формулювання:
"зазвичай потрібно..."
"приблизно потрібно..."
"я рекомендую взяти..."

якщо такої кількості немає в базі рецепта.

Якщо точної кількості немає у рецепті, чесно скажи:
"У моїй базі для цього рецепта точна кількість цього інгредієнта не вказана."

 === 8. БЕЗПЕКА ТА ОБМЕЖЕННЯ ===

 ChefBot не є лікарем або дієтологом.
 Не став медичних діагнозів і не створюй лікувальні дієти.
 Якщо користувач повідомляє про серйозну алергію:
 - не гарантуй абсолютну безпечність продукту;
 - рекомендуй перевірити склад і маркування конкретного продукту.
 Не представляй кулінарні рекомендації як медичні поради.

 === 9. ЗАПИТИ ПОЗА ТЕМОЮ ===

 Якщо запит не стосується:
 - їжі;
 - рецептів;
 - продуктів;
 - приготування;
 - кулінарних замін;
 - кулінарних одиниць вимірювання, ввічливо поясни, що ChefBot спеціалізується на кулінарних питаннях.
 Не намагайся бути універсальним асистентом.

 === 10. СТИЛЬ ВІДПОВІДІ ===
 Відповідай:
 - просто;
 - доброзичливо;
 - конкретно;
 - без зайвої теорії.

 Для рецепта бажано показувати:
 - назву страви;
 - інгредієнти;
 - короткі кроки приготування.

 Для конвертації:
 - одразу показуй точний результат інструмента.

 Для заміни:
 - чітко покажи знайдені альтернативи.

 Головний принцип ChefBot:
 Мовна модель розуміє запит і пояснює результат.
 Спеціалізовані інструменти виконують пошук і точні операції.
 Якщо існує відповідний інструмент — використовуй його.

"""

agent = create_agent(model=chat_model, tools=tools, system_prompt=SYSTEM_PROMPT)
print("✅ Агент ChefBot створено.")

# Тест
test_msgs = [{"role":"user","content":"Що приготувати з курки та картоплі?"}]
r = agent.invoke({"messages": test_msgs})
print("🤖", r["messages"][-1].content[:150])

✅ Агент ChefBot створено.
🤖 Ось кілька ідей страв, які можна приготувати з курки та картоплі:

### 1. Курка з картоплею
- **Інгредієнти**: куряче філе, картопля, морква, олія, сі


## 5. Інтерактивний чат

In [ ]:
def run_chefbot_session():
    print("\n" + "=" * 60)
    print("🤖 ChefBot")
    print("=" * 60)
    print("exit — завершити, /reset — скинути контекст")
    print("-" * 60)
    messages = []
    while True:
        try:
            user_input = input("\n👤 Ви: ").strip()
            if not user_input: continue
            if user_input.lower() in ("exit","вихід","/exit","quit"):
                print("👋 До побачення!"); break
            if user_input.lower() in ("/reset","reset"):
                messages = []; print("🔄 Контекст скинуто."); continue
            messages.append({"role":"user","content":user_input})
            print("🤔 Думаю...")
            result = agent.invoke({"messages": messages})
            messages = result["messages"]
            last = messages[-1]
            answer = last.content if hasattr(last,"content") else last.get("content","")
            print(f"\n🤖 {answer}")
        except KeyboardInterrupt:
            print("\n👋 Перервано."); break
        except Exception as e:
            print(f"❌ Помилка: {e}")

run_chefbot_session()


🤖 ChefBot
exit — завершити, /reset — скинути контекст
------------------------------------------------------------

👤 Ви: що приготувати на сніданок
🤔 Думаю...

🤖 Ось кілька ідей для сніданку:

1. **Омлет**
   - **Інгредієнти:** яйця, молоко, сіль, олія
   - **Час приготування:** 10 хвилин
   - **Порцій:** 2
   - **Приготування:** 
     1. Збийте яйця з молоком і сіллю.
     2. Розігрійте пательню з олією.
     3. Вилийте суміш і готуйте під кришкою.

2. **Млинці**
   - **Інгредієнти:** борошно, молоко, яйця, цукор, сіль, олія
   - **Час приготування:** 30 хвилин
   - **Порцій:** 4
   - **Приготування:** 
     1. Змішайте яйця, молоко, цукор і сіль.
     2. Поступово додайте борошно.
     3. Додайте трохи олії.
     4. Смажте тонкі млинці з обох боків.

3. **Рисова каша**
   - **Інгредієнти:** рис, молоко, вода, цукор, сіль, масло
   - **Час приготування:** 35 хвилин
   - **Порцій:** 3
   - **Приготування:** 
     1. Промийте рис.
     2. Варіть його у воді 10 хвилин.
     3. Додайте 

## 6. Автоматичне тестування

In [ ]:
def run_automatic_tests():
    test_queries = [
        "Привіт! Що ти вмієш?",
        "Що можна приготувати з курки та картоплі?",
        "Скільки грамів борошна в одній склянці?",
        "Чим замінити яйця у випічці?",
        "Запам'ятай, що я не їм глютен.",
        "Знайди мені рецепт на вечерю.",
        "Скільки коштує борошно?",
        "Яка погода в Києві?",
        "Знайди мені швидкий сніданок.",
        "Дай рецепт без глютену.",
    ]
    messages = []
    print("\n" + "="*60 + "\n🧪 ТЕСТУВАННЯ ChefBot\n" + "="*60)
    for i, q in enumerate(test_queries, 1):
        print(f"\nТест #{i}\n" + "-"*60 + f"\n👤 {q}")
        messages.append({"role":"user","content":q})
        try:
            r = agent.invoke({"messages": messages})
            messages = r["messages"]
            last = messages[-1]
            a = last.content if hasattr(last,'content') else last.get('content','')
            print(f"✅ {a}")
        except Exception as e:
            print(f"❌ {e}")
    print("\n✅ Тестування завершено.")

run_automatic_tests()


🧪 ТЕСТУВАННЯ ChefBot

Тест #1
------------------------------------------------------------
👤 Привіт! Що ти вмієш?
✅ Привіт! Я можу допомогти знайти рецепти, підібрати страви за наявними інгредієнтами, конвертувати кулінарні одиниці вимірювання та знаходити замінники для інгредієнтів. Як можу допомогти тобі сьогодні?

Тест #2
------------------------------------------------------------
👤 Що можна приготувати з курки та картоплі?
✅ Ось кілька страв, які можна приготувати з курки та картоплі:

1. **Курка з картоплею**
   - **Інгредієнти:** куряче філе, картопля, морква, олія, сіль, перець
   - **Час приготування:** 55 хвилин
   - **Кількість порцій:** 4
   - **Приготування:** Наріжте курку, картоплю та моркву. Додайте олію, сіль і перець. Запікайте при 190°C приблизно 40–45 хвилин.

2. **Борщ**
   - **Інгредієнти:** буряк, капуста, картопля, морква, цибуля, томатна паста
   - **Час приготування:** 90 хвилин
   - **Кількість порцій:** 6
   - **Приготування:** Підготуйте та наріжте овочі. 

---
## Інтерфейс

In [ ]:
%%writefile app.py

import streamlit as st

st.title("ChefBot")
st.write("Ваш персональний кулінарний AI-асистент")

Writing app.py


In [ ]:
!pip install -q streamlit

In [ ]:
%%writefile app.py

import streamlit as st
import os
import re

from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

RECIPES_DB = {
    "борщ": {
        "ingredients": [
            "буряк — 300 г",
            "капуста — 300 г",
            "картопля — 400 г",
            "морква — 150 г",
            "цибуля — 100 г",
            "томатна паста — 2 ст.л.",
            "вода — 2 л",
            "олія — 1 ст.л.",
            "сіль — за смаком",
        ],
        "time_minutes": 90,
        "servings": 6,
        "instructions": (
            "1. Підготуйте та наріжте овочі. "
            "2. Відваріть картоплю й капусту. "
            "3. Додайте тушковані буряк, моркву, цибулю та томатну пасту. "
            "4. Варіть до готовності."
        ),
        "tags": [
            "перша страва",
            "суп",
            "обід",
            "українська кухня",
            "вегетаріанське",
            "овочеве",
        ],
        "gluten_free": True,
    },

    "омлет": {
        "ingredients": [
            "яйця — 4 шт.",
            "молоко — 100 мл",
            "сіль — за смаком",
            "олія — 1 ч.л.",
        ],
        "time_minutes": 10,
        "servings": 2,
        "instructions": (
            "1. Збийте яйця з молоком і сіллю. "
            "2. Розігрійте пательню з олією. "
            "3. Вилийте суміш і готуйте під кришкою."
        ),
        "tags": [
            "сніданок",
            "швидко",
            "білкове",
            "вегетаріанське",
        ],
        "gluten_free": True,
    },

    "курка з картоплею": {
        "ingredients": [
            "куряче філе — 500 г",
            "картопля — 700 г",
            "морква — 150 г",
            "олія — 2 ст.л.",
            "сіль — за смаком",
            "перець — за смаком",
        ],
        "time_minutes": 55,
        "servings": 4,
        "instructions": (
            "1. Наріжте курку, картоплю та моркву. "
            "2. Додайте олію, сіль і перець. "
            "3. Запікайте при 190°C приблизно 40–45 хвилин."
        ),
        "tags": [
            "основна страва",
            "обід",
            "вечеря",
            "білкове",
            "духовка",
            "курка",
        ],
        "gluten_free": True,
    },

    "вареники з картоплею": {
        "ingredients": [
            "борошно — 500 г",
            "вода — 250 мл",
            "яйця — 1 шт.",
            "картопля — 700 г",
            "цибуля — 150 г",
            "олія — 1 ст.л.",
            "сіль — за смаком",
        ],
        "time_minutes": 80,
        "servings": 5,
        "instructions": (
            "1. Замісіть тісто з борошна, води, яйця та солі. "
            "2. Приготуйте начинку з картоплі й цибулі. "
            "3. Сформуйте вареники. "
            "4. Варіть у підсоленій воді 4–5 хвилин після спливання."
        ),
        "tags": [
            "основна страва",
            "обід",
            "українська кухня",
            "вегетаріанське",
            "тісто",
        ],
        "gluten_free": False,
    },

    "млинці": {
        "ingredients": [
            "борошно — 250 г",
            "молоко — 500 мл",
            "яйця — 2 шт.",
            "цукор — 30 г",
            "сіль — 1 щіпка",
            "олія — 2 ст.л.",
        ],
        "time_minutes": 30,
        "servings": 4,
        "instructions": (
            "1. Змішайте яйця, молоко, цукор і сіль. "
            "2. Поступово додайте борошно. "
            "3. Додайте трохи олії. "
            "4. Смажте тонкі млинці з обох боків."
        ),
        "tags": [
            "сніданок",
            "десерт",
            "солодке",
            "вегетаріанське",
            "швидко",
        ],
        "gluten_free": False,
    },

    "овочевий салат": {
        "ingredients": [
            "помідори — 250 г",
            "огірки — 200 г",
            "болгарський перець — 150 г",
            "листя салату — 80 г",
            "олія — 1 ст.л.",
            "сіль — за смаком",
        ],
        "time_minutes": 10,
        "servings": 2,
        "instructions": (
            "1. Помийте та наріжте овочі. "
            "2. Додайте листя салату. "
            "3. Заправте олією та посоліть."
        ),
        "tags": [
            "салат",
            "швидко",
            "вегетаріанське",
            "легка страва",
            "овочеве",
            "обід",
            "вечеря",
        ],
        "gluten_free": True,
    },

    "рибний суп": {
        "ingredients": [
            "риба — 400 г",
            "картопля — 300 г",
            "морква — 100 г",
            "цибуля — 100 г",
            "вода — 1.5 л",
            "сіль — за смаком",
            "зелень — 20 г",
        ],
        "time_minutes": 45,
        "servings": 4,
        "instructions": (
            "1. Наріжте овочі та відваріть їх до напівготовності. "
            "2. Додайте шматочки риби. "
            "3. Варіть ще 10–15 хвилин. "
            "4. Додайте сіль і зелень."
        ),
        "tags": [
            "перша страва",
            "суп",
            "рибне",
            "обід",
            "білкове",
        ],
        "gluten_free": True,
    },

    "овочеве рагу": {
        "ingredients": [
            "кабачок — 300 г",
            "картопля — 400 г",
            "морква — 150 г",
            "болгарський перець — 150 г",
            "помідори — 250 г",
            "олія — 2 ст.л.",
            "сіль — за смаком",
        ],
        "time_minutes": 40,
        "servings": 4,
        "instructions": (
            "1. Наріжте всі овочі. "
            "2. Злегка протушкуйте моркву та картоплю. "
            "3. Додайте інші овочі. "
            "4. Тушкуйте під кришкою до готовності."
        ),
        "tags": [
            "основна страва",
            "обід",
            "вечеря",
            "вегетаріанське",
            "овочеве",
            "тушковане",
        ],
        "gluten_free": True,
    },

    "яблучний пиріг": {
        "ingredients": [
            "борошно — 250 г",
            "яблука — 500 г",
            "яйця — 3 шт.",
            "цукор — 150 г",
            "масло — 100 г",
            "розпушувач — 10 г",
        ],
        "time_minutes": 60,
        "servings": 8,
        "instructions": (
            "1. Збийте яйця з цукром. "
            "2. Додайте розтоплене масло, борошно та розпушувач. "
            "3. Додайте нарізані яблука. "
            "4. Випікайте при 180°C приблизно 40 хвилин."
        ),
        "tags": [
            "десерт",
            "випічка",
            "солодке",
            "духовка",
            "вегетаріанське",
        ],
        "gluten_free": False,
    },

    "рисова каша": {
        "ingredients": [
            "рис — 180 г",
            "молоко — 500 мл",
            "вода — 250 мл",
            "цукор — 30 г",
            "сіль — 1 щіпка",
            "масло — 20 г",
        ],
        "time_minutes": 35,
        "servings": 3,
        "instructions": (
            "1. Промийте рис. "
            "2. Варіть його у воді 10 хвилин. "
            "3. Додайте молоко, цукор і сіль. "
            "4. Варіть до м'якості та додайте масло."
        ),
        "tags": [
            "сніданок",
            "каша",
            "солодке",
            "вегетаріанське",
        ],
        "gluten_free": True,
    },

    "гречка з куркою": {
        "ingredients": [
            "гречка — 200 г",
            "куряче філе — 400 г",
            "морква — 100 г",
            "цибуля — 100 г",
            "олія — 1 ст.л.",
            "сіль — за смаком",
        ],
        "time_minutes": 40,
        "servings": 4,
        "instructions": (
            "1. Наріжте та приготуйте куряче філе. "
            "2. Додайте моркву й цибулю. "
            "3. Окремо відваріть гречку. "
            "4. Змішайте всі інгредієнти."
        ),
        "tags": [
            "основна страва",
            "обід",
            "вечеря",
            "білкове",
            "швидко",
            "курка",
        ],
        "gluten_free": True,
    },

    "салат з тунцем": {
        "ingredients": [
            "тунець — 160 г",
            "яйця — 2 шт.",
            "огірки — 150 г",
            "помідори — 200 г",
            "листя салату — 80 г",
            "олія — 1 ст.л.",
        ],
        "time_minutes": 15,
        "servings": 2,
        "instructions": (
            "1. Відваріть і наріжте яйця. "
            "2. Наріжте овочі. "
            "3. Додайте тунець і листя салату. "
            "4. Заправте олією."
        ),
        "tags": [
            "салат",
            "обід",
            "вечеря",
            "білкове",
            "швидко",
            "рибне",
            "легка страва",
        ],
        "gluten_free": True,
    },

    "паста з куркою у вершковому соусі": {
        "ingredients": [
            "паста — 250 г",
            "куряче філе — 350 г",
            "вершки — 200 мл",
            "часник — 2 зубчики",
            "олія — 1 ст.л.",
            "сіль — за смаком",
            "перець — за смаком",
        ],
        "time_minutes": 30,
        "servings": 3,
        "instructions": (
            "1. Відваріть пасту відповідно до інструкції на упаковці. "
            "2. Наріжте куряче філе та обсмажте на олії. "
            "3. Додайте подрібнений часник і вершки. "
            "4. Тушкуйте соус кілька хвилин. "
            "5. Додайте пасту, перемішайте та прогрійте."
        ),
        "tags": [
            "основна страва",
            "обід",
            "вечеря",
            "паста",
            "курка",
            "білкове",
            "швидко",
        ],
        "gluten_free": False,
    },

    "запечений лосось з овочами": {
        "ingredients": [
            "лосось — 350 г",
            "кабачок — 200 г",
            "болгарський перець — 150 г",
            "помідори — 200 г",
            "олія — 1 ст.л.",
            "лимон — 0.5 шт.",
            "сіль — за смаком",
            "перець — за смаком",
        ],
        "time_minutes": 35,
        "servings": 2,
        "instructions": (
            "1. Наріжте овочі та викладіть у форму для запікання. "
            "2. Додайте олію, сіль і перець. "
            "3. Покладіть зверху лосось і скропіть лимонним соком. "
            "4. Запікайте при 190°C приблизно 20–25 хвилин."
        ),
        "tags": [
            "основна страва",
            "обід",
            "вечеря",
            "рибне",
            "білкове",
            "духовка",
            "легка страва",
        ],
        "gluten_free": True,
    },

    "рис з куркою та овочами": {
        "ingredients": [
            "рис — 200 г",
            "куряче філе — 400 г",
            "морква — 100 г",
            "болгарський перець — 150 г",
            "кабачок — 200 г",
            "олія — 1 ст.л.",
            "сіль — за смаком",
            "перець — за смаком",
        ],
        "time_minutes": 35,
        "servings": 4,
        "instructions": (
            "1. Відваріть рис до готовності. "
            "2. Наріжте куряче філе та обсмажте на олії. "
            "3. Додайте нарізані овочі та готуйте до м'якості. "
            "4. Додайте рис, сіль і перець та перемішайте."
        ),
        "tags": [
            "основна страва",
            "обід",
            "вечеря",
            "курка",
            "білкове",
            "швидко",
        ],
        "gluten_free": True,
    },

    "шакшука": {
        "ingredients": [
            "яйця — 4 шт.",
            "помідори — 400 г",
            "болгарський перець — 150 г",
            "цибуля — 100 г",
            "часник — 2 зубчики",
            "олія — 1 ст.л.",
            "сіль — за смаком",
        ],
        "time_minutes": 25,
        "servings": 2,
        "instructions": (
            "1. Наріжте цибулю та перець і обсмажте на олії. "
            "2. Додайте помідори й часник та тушкуйте кілька хвилин. "
            "3. Зробіть заглиблення у соусі та розбийте в них яйця. "
            "4. Готуйте під кришкою до бажаної готовності яєць."
        ),
        "tags": [
            "сніданок",
            "швидко",
            "вегетаріанське",
            "білкове",
            "яйця",
        ],
        "gluten_free": True,
    },

    "крем-суп з кабачка": {
        "ingredients": [
            "кабачок — 500 г",
            "картопля — 250 г",
            "морква — 100 г",
            "цибуля — 100 г",
            "вершки — 150 мл",
            "вода — 800 мл",
            "сіль — за смаком",
            "олія — 1 ст.л.",
        ],
        "time_minutes": 35,
        "servings": 4,
        "instructions": (
            "1. Наріжте овочі. "
            "2. Відваріть їх у воді до м'якості. "
            "3. Подрібніть овочі блендером до однорідності. "
            "4. Додайте вершки, сіль і прогрійте суп ще кілька хвилин."
        ),
        "tags": [
            "перша страва",
            "суп",
            "обід",
            "вегетаріанське",
            "овочеве",
            "легка страва",
        ],
        "gluten_free": True,
    },

    "спагеті з томатним соусом": {
        "ingredients": [
            "спагеті — 250 г",
            "помідори — 400 г",
            "томатна паста — 1 ст.л.",
            "часник — 2 зубчики",
            "олія — 1 ст.л.",
            "сіль — за смаком",
            "зелень — 20 г",
        ],
        "time_minutes": 25,
        "servings": 3,
        "instructions": (
            "1. Відваріть спагеті до готовності. "
            "2. На олії коротко обсмажте часник. "
            "3. Додайте помідори та томатну пасту і тушкуйте 10 хвилин. "
            "4. Змішайте соус зі спагеті та додайте зелень."
        ),
        "tags": [
            "основна страва",
            "обід",
            "вечеря",
            "паста",
            "вегетаріанське",
            "швидко",
        ],
        "gluten_free": False,
    },

    "запечені овочі з фетою": {
        "ingredients": [
            "кабачок — 300 г",
            "болгарський перець — 200 г",
            "помідори — 300 г",
            "фета — 150 г",
            "олія — 1.5 ст.л.",
            "сіль — за смаком",
            "зелень — 20 г",
        ],
        "time_minutes": 35,
        "servings": 3,
        "instructions": (
            "1. Наріжте овочі та викладіть у форму. "
            "2. Додайте олію та сіль. "
            "3. Запікайте при 190°C приблизно 20 хвилин. "
            "4. Додайте фету і запікайте ще 5–10 хвилин. "
            "5. Перед подачею додайте зелень."
        ),
        "tags": [
            "основна страва",
            "обід",
            "вечеря",
            "вегетаріанське",
            "овочеве",
            "духовка",
            "легка страва",
        ],
        "gluten_free": True,
    },

    "бананові панкейки": {
        "ingredients": [
            "банан — 1 шт. (приблизно 120 г)",
            "яйця — 2 шт.",
            "борошно — 100 г",
            "молоко — 100 мл",
            "олія — 1 ч.л.",
        ],
        "time_minutes": 20,
        "servings": 2,
        "instructions": (
            "1. Розімніть банан виделкою. "
            "2. Додайте яйця та молоко і перемішайте. "
            "3. Додайте борошно та перемішайте до однорідності. "
            "4. Смажте невеликі панкейки на пательні з обох боків."
        ),
        "tags": [
            "сніданок",
            "десерт",
            "солодке",
            "швидко",
            "вегетаріанське",
        ],
        "gluten_free": False,
    },
}

UNIT_CONVERSIONS = {
    # Склянки → грами для конкретних продуктів
    ("склянка", "г", "борошно"): 150,
    ("склянка", "г", "цукор"): 200,
    ("склянка", "г", "рис"): 180,
    ("склянка", "г", "гречка"): 170,
    ("склянка", "г", "вівсянка"): 90,

    # Склянки → мілілітри
    ("склянка", "мл", ""): 240,

    # Столові ложки → мілілітри
    ("ст.л.", "мл", ""): 15,

    # Чайні ложки → мілілітри
    ("ч.л.", "мл", ""): 5,

    # Столові ложки → грами
    ("ст.л.", "г", "борошно"): 10,
    ("ст.л.", "г", "цукор"): 20,
    ("ст.л.", "г", "масло"): 15,
    ("ст.л.", "г", "олія"): 14,

    # Чайні ложки → грами
    ("ч.л.", "г", "сіль"): 7,
    ("ч.л.", "г", "цукор"): 5,
    ("ч.л.", "г", "олія"): 5,
}


SUBSTITUTIONS = {
    "яйця": [
        {
            "замінник": "банан — половина стиглого банана замість 1 яйця",
            "для": "солодка випічка, панкейки",
            "примітка": "додає солодкість, вологість і легкий банановий смак",
        },
        {
            "замінник": "льняне насіння — 1 ст.л. меленого насіння + 3 ст.л. води",
            "для": "випічка, панкейки",
            "примітка": "залиш суміш приблизно на 10 хвилин до загущення",
        },
        {
            "замінник": "аквафаба — 3 ст.л. замість 1 яйця",
            "для": "меренги, муси, легка випічка",
            "примітка": "аквафаба — це рідина з-під консервованого нуту",
        },
    ],

    "молоко": [
        {
            "замінник": "вівсяний напій",
            "для": "каші, млинці, випічка, соуси",
            "примітка": "має нейтральний смак; при безглютеновій дієті перевір маркування",
        },
        {
            "замінник": "соєвий напій",
            "для": "омлети, випічка, соуси, каші",
            "примітка": "за текстурою часто близький до звичайного молока, але містить сою",
        },
        {
            "замінник": "мигдальний напій",
            "для": "каші, десерти, випічка",
            "примітка": "може мати легкий горіховий смак",
        },
        {
            "замінник": "кокосове молоко",
            "для": "каші, десерти, соуси",
            "примітка": "має виражений кокосовий смак і може бути жирнішим",
        },
    ],

    "вершки": [
        {
            "замінник": "рослинні кулінарні вершки",
            "для": "вершкові соуси, паста, крем-супи",
            "примітка": "найближча заміна за текстурою; обирай продукт саме для готування",
        },
        {
            "замінник": "кокосові вершки",
            "для": "соуси, супи, десерти",
            "примітка": "добре загущують, але можуть додати кокосовий смак",
        },
        {
            "замінник": "молоко або рослинний напій + трохи крохмалю",
            "для": "соуси та крем-супи",
            "примітка": "сама рідина значно рідша за вершки, тому для густоти потрібен крохмаль",
        },
    ],

    "борошно": [
        {
            "замінник": "рисове борошно",
            "для": "млинці, печиво, безглютенова випічка",
            "примітка": "може зробити готовий виріб більш крихким",
        },
        {
            "замінник": "вівсяне борошно",
            "для": "млинці, панкейки, випічка",
            "примітка": "для безглютенового рецепта потрібно перевірити відповідне маркування",
        },
        {
            "замінник": "кукурудзяне борошно",
            "для": "коржі, панкейки, несолодка випічка",
            "примітка": "змінює структуру, колір і смак тіста",
        },
    ],

    "масло": [
        {
            "замінник": "рослинна олія",
            "для": "випічка, смаження",
            "примітка": "100 г масла можна орієнтовно замінити приблизно 80 мл олії",
        },
        {
            "замінник": "кокосова олія",
            "для": "десерти, випічка",
            "примітка": "може додати кокосовий аромат",
        },
        {
            "замінник": "яблучне пюре",
            "для": "солодка випічка",
            "примітка": "зменшує жирність і робить випічку більш вологою",
        },
    ],

    "цукор": [
        {
            "замінник": "мед",
            "для": "десерти, напої, випічка",
            "примітка": "мед додає рідину, тому в деяких рецептах потрібно зменшити інші рідини",
        },
        {
            "замінник": "стиглий банан",
            "для": "каші, панкейки, солодка випічка",
            "примітка": "додає власний смак і змінює текстуру",
        },
        {
            "замінник": "еритритол",
            "для": "напої, десерти, деякі види випічки",
            "примітка": "солодкість відрізняється від цукру, тому пропорцію потрібно коригувати",
        },
    ],

    "сметана": [
        {
            "замінник": "натуральний йогурт",
            "для": "соуси, заправки, випічка",
            "примітка": "може бути менш жирним і трохи кислішим",
        },
        {
            "замінник": "грецький йогурт",
            "для": "соуси, заправки, подача до страв",
            "примітка": "має густу текстуру, близьку до сметани",
        },
        {
            "замінник": "рослинний йогурт без цукру",
            "для": "соуси та заправки",
            "примітка": "підійде для безлактозного варіанта, якщо смак нейтральний",
        },
    ],

    "фета": [
        {
            "замінник": "бринза",
            "для": "салати, запечені овочі",
            "примітка": "може бути солонішою, тому обережно додавай сіль",
        },
        {
            "замінник": "козячий сир",
            "для": "салати, запечені овочі",
            "примітка": "має більш виражений смак",
        },
        {
            "замінник": "рослинний сир типу фета",
            "для": "салати, запечені овочі",
            "примітка": "можна використати для безлактозного або веганського варіанта",
        },
    ],

    "рис": [
        {
            "замінник": "гречка",
            "для": "гарнір, страви з куркою та овочами",
            "примітка": "має більш виражений смак і іншу текстуру",
        },
        {
            "замінник": "кіноа",
            "для": "гарніри, салати",
            "примітка": "текстура та час приготування відрізняються від рису",
        },
        {
            "замінник": "булгур",
            "для": "гарнір, страви з овочами",
            "примітка": "містить глютен",
        },
    ],

    "куряче філе": [
        {
            "замінник": "філе індички",
            "для": "запікання, смаження, страви з рисом або пастою",
            "примітка": "найближча заміна за способом приготування",
        },
        {
            "замінник": "тофу",
            "для": "страви з овочами, рисом, салати",
            "примітка": "вегетаріанська альтернатива з іншою текстурою та смаком",
        },
    ],

    "лосось": [
        {
            "замінник": "форель",
            "для": "запікання, смаження",
            "примітка": "дуже близька за жирністю та способом приготування",
        },
        {
            "замінник": "дорадо",
            "для": "запікання",
            "примітка": "менш жирна риба, тому результат буде трохи іншим",
        },
        {
            "замінник": "хек",
            "для": "запікання, тушкування",
            "примітка": "значно менш жирний за лосось",
        },
    ],

    "помідори": [
        {
            "замінник": "консервовані томати",
            "для": "соуси, супи, шакшука",
            "примітка": "добре підходять для термічної обробки",
        },
        {
            "замінник": "томатна пасата",
            "для": "соуси, супи",
            "примітка": "має більш однорідну текстуру",
        },
    ],
}

@tool
def recipe_search(query: str) -> str:
    """
    Шукає рецепти за назвою, інгредієнтами, тегами або харчовими обмеженнями.

    Використовуй цей інструмент, коли користувач:
    - просить рецепт конкретної страви;
    - хоче знайти страву з наявних продуктів;
    - шукає страву певної категорії;
    - просить рецепт без глютену.

    Повертає до трьох найбільш релевантних рецептів:
    назву, інгредієнти, час приготування, кількість порцій
    та коротку інструкцію.
    """
    if not isinstance(query, str) or not query.strip():
        return "Запит порожній. Вкажи назву страви, інгредієнти або категорію."

    query_lower = query.lower().strip()
    query_words = [
        word
        for word in re.findall(r"[а-яіїєґa-z0-9'-]+", query_lower)
        if len(word) > 2
    ]

    results = []

    for name, recipe in RECIPES_DB.items():
        score = 0
        match_reasons = []

        # Точний або частковий збіг за назвою
        if query_lower in name.lower() or name.lower() in query_lower:
            score += 5
            match_reasons.append("назва")

        # Пошук за інгредієнтами
        matched_ingredients = []
        for ingredient in recipe["ingredients"]:
            ingredient_lower = ingredient.lower()

            if any(
                word in ingredient_lower or ingredient_lower in word
                for word in query_words
            ):
                matched_ingredients.append(ingredient)

        if matched_ingredients:
            score += len(matched_ingredients) * 2
            match_reasons.append(
                f"інгредієнти: {', '.join(matched_ingredients)}"
            )

        # Пошук за тегами
        matched_tags = []
        for tag in recipe["tags"]:
            tag_lower = tag.lower()

            if any(
                word in tag_lower or tag_lower in word
                for word in query_words
            ):
                matched_tags.append(tag)

        if matched_tags:
            score += len(matched_tags)
            match_reasons.append(f"теги: {', '.join(matched_tags)}")

        # Окрема перевірка запиту без глютену
        gluten_free_request = (
            "без глютену" in query_lower
            or "безглютен" in query_lower
            or "глютен free" in query_lower
        )

        if gluten_free_request:
            if recipe["gluten_free"]:
                score += 4
                match_reasons.append("без глютену")
            else:
                # Рецепти з глютеном не повертаємо
                continue

        if score > 0:
            results.append(
                {
                    "name": name,
                    "recipe": recipe,
                    "score": score,
                    "reason": "; ".join(match_reasons),
                }
            )

    if not results:
        return (
            f"У локальній базі не знайдено рецептів за запитом «{query}». "
            "Спробуй назвати інший продукт, страву або категорію."
        )

    # Найбільш релевантні рецепти показуємо першими
    results.sort(key=lambda item: item["score"], reverse=True)

    output = []

    for item in results[:3]:
        name = item["name"]
        recipe = item["recipe"]

        output.append(f"🍽️ {name.capitalize()}")
        output.append(
            f"Інгредієнти: {', '.join(recipe['ingredients'])}"
        )
        output.append(
            f"Час: {recipe['time_minutes']} хвилин | "
            f"Порцій: {recipe['servings']}"
        )
        output.append(
            f"Без глютену: {'так' if recipe['gluten_free'] else 'ні'}"
        )
        output.append(f"Знайдено за: {item['reason']}")
        output.append(f"Приготування: {recipe['instructions']}")
        output.append("")

    return "\n".join(output).strip()


    @tool
def unit_converter(query: str) -> str:
    """
    Конвертує кулінарні одиниці вимірювання на основі локальної таблиці.

    Використовуй цей інструмент, коли користувач просить:
    - перевести склянки в грами або мілілітри;
    - перевести столові чи чайні ложки в грами або мілілітри;
    - визначити вагу певної кількості продукту.

    Приклади:
    - "скільки грамів у склянці борошна";
    - "2 склянки цукру — це скільки грамів";
    - "3 столові ложки — це скільки мілілітрів".

    Повертає результат лише для конвертацій, які є в UNIT_CONVERSIONS.
    """
    if not isinstance(query, str) or not query.strip():
        return "Запит порожній. Вкажи кількість, одиницю та потрібний формат."

    query_lower = query.lower().strip()

    # Визначення кількості
    numbers = re.findall(r"\d+(?:[.,]\d+)?", query_lower)

    try:
        amount = float(numbers[0].replace(",", ".")) if numbers else 1.0
    except (ValueError, IndexError):
        return "Не вдалося визначити кількість. Спробуй вказати число, наприклад 2."

    if amount <= 0:
        return "Кількість повинна бути більшою за нуль."

    # Визначення початкової одиниці
    from_unit = None

    if "склянк" in query_lower:
        from_unit = "склянка"
    elif (
        "ст.л." in query_lower
        or "столов" in query_lower
        or "столова лож" in query_lower
    ):
        from_unit = "ст.л."
    elif (
        "ч.л." in query_lower
        or "чайн" in query_lower
        or "чайна лож" in query_lower
    ):
        from_unit = "ч.л."

    # Визначення цільової одиниці
    to_unit = None

    if "грам" in query_lower or re.search(r"\bг\b", query_lower):
        to_unit = "г"
    elif "мілілітр" in query_lower or re.search(r"\bмл\b", query_lower):
        to_unit = "мл"

    # Визначення продукту
    product = ""

    known_products = [
        "борошно",
        "цукор",
        "рис",
        "гречка",
        "сіль",
        "масло",
    ]

    for known_product in known_products:
        if known_product in query_lower:
            product = known_product
            break

    if not from_unit:
        return (
            "Не вдалося визначити початкову одиницю. "
            "Використай слова «склянка», «столова ложка» або «чайна ложка»."
        )

    if not to_unit:
        return (
            "Не вдалося визначити цільову одиницю. "
            "Вкажи, чи потрібні грами або мілілітри."
        )

    specific_key = (from_unit, to_unit, product)
    universal_key = (from_unit, to_unit, "")

    if product and specific_key in UNIT_CONVERSIONS:
        conversion_value = UNIT_CONVERSIONS[specific_key]
        result = amount * conversion_value

        return (
            f"{amount:g} {from_unit} продукту «{product}» "
            f"≈ {result:.1f} {to_unit}."
        )

    if universal_key in UNIT_CONVERSIONS:
        conversion_value = UNIT_CONVERSIONS[universal_key]
        result = amount * conversion_value

        return f"{amount:g} {from_unit} ≈ {result:.1f} {to_unit}."

    if not product and to_unit == "г":
        return (
            "Для переведення в грами потрібно вказати продукт, "
            "оскільки різні продукти мають різну щільність."
        )

    return (
        f"У локальній таблиці немає конвертації з «{from_unit}» "
        f"у «{to_unit}» для продукту «{product or 'не вказано'}»."
    )


@tool
def substitution_finder(ingredient: str) -> str:
    """
    Шукає кулінарні замінники для конкретного інгредієнта.

    Використовуй цей інструмент, коли користувач:
    - питає, чим замінити інгредієнт;
    - не має певного продукту;
    - хоче адаптувати рецепт;
    - має харчове обмеження.

    Приклади:
    - "чим замінити яйця";
    - "що використати замість молока";
    - "у мене немає сметани".

    Повертає доступні заміни, сферу використання та важливі примітки.
    """
    if not isinstance(ingredient, str) or not ingredient.strip():
        return "Вкажи назву інгредієнта, який потрібно замінити."

    ingredient_lower = ingredient.lower().strip()

    found_key = None

    for key in SUBSTITUTIONS:
        if key in ingredient_lower or ingredient_lower in key:
            found_key = key
            break

    if not found_key:
        return (
            f"У локальній базі не знайдено замінника для «{ingredient}». "
            "Спробуй вказати коротку назву продукту, наприклад «молоко»."
        )

    result = [f"Замінники для «{found_key}»:"]

    for substitute in SUBSTITUTIONS[found_key]:
        result.append(
            f"• {substitute['замінник']}. "
            f"Підходить для: {substitute['для']}. "
            f"Примітка: {substitute['примітка']}."
        )

    return "\n".join(result)

